<a href="https://colab.research.google.com/github/niniqoiii/Springfield/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Springfield Identity — Training Notebook

This notebook trains a CNN from scratch for Simpsons character classification.

It will:
1. Download `Simpsons.zip` from Google Drive
2. Extract the dataset
3. Find `characters_train`
4. Train a CNN from scratch
5. Train for 25 epochs and save the best `model.pth`, `simpson_model.pth`, and `classes.json`

In [ ]:
# ============================================================================
# STEP 1: Install / Import Libraries
# ============================================================================
!pip -q install gdown

import os
import json
import random
import zipfile
import shutil
from pathlib import Path

import gdown
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from PIL import Image

print("✓ Libraries imported")

✓ Libraries imported


In [ ]:
# ============================================================================
# STEP 2: Reproducibility — Set Random Seeds
# ============================================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("✓ Random seeds set")

✓ Random seeds set


In [ ]:
# ============================================================================
# STEP 3: Download and Extract Dataset
# ============================================================================
# Google Drive link:
# https://drive.google.com/file/d/12FqLnmk6Cu0NR8UehRQiD7az2iDYufCX/view?usp=sharing

file_id = "12FqLnmk6Cu0NR8UehRQiD7az2iDYufCX"
zip_file = "Simpsons.zip"
extract_dir = "/content/simpsons_extracted"

# Clean old extraction to avoid wrong/duplicate paths from previous runs
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)

print("Downloading dataset from Google Drive...")
gdown.download(f"https://drive.google.com/uc?id={file_id}", zip_file, quiet=False)
print("✓ Download complete")

print("Extracting dataset...")
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("✓ Extraction complete")

Downloading...
From (original): https://drive.google.com/uc?id=12FqLnmk6Cu0NR8UehRQiD7az2iDYufCX
From (redirected): https://drive.google.com/uc?id=12FqLnmk6Cu0NR8UehRQiD7az2iDYufCX&confirm=t&uuid=64dc286d-b6f9-4bc4-a5a8-ebf0fb887389
To: /content/Simpsons.zip
100%|██████████| 432M/432M [00:06<00:00, 71.1MB/s]


✓ Download complete
Extracting dataset...
✓ Extraction complete


In [ ]:
# ============================================================================
# STEP 4: Locate Training Data Directory
# ============================================================================
print("Locating training directory...")

train_dir = None
for root, dirs, files in os.walk(extract_dir):
    if "__MACOSX" in root:
        continue
    if "characters_train" in dirs:
        candidate = os.path.join(root, "characters_train")
        image_count = 0
        for r, d, f in os.walk(candidate):
            image_count += sum(x.lower().endswith((".jpg", ".jpeg", ".png")) for x in f)
        if image_count > 0:
            train_dir = candidate
            break

if train_dir is None:
    raise FileNotFoundError("Could not find a valid characters_train directory with images!")

print(f"✓ Training directory found: {train_dir}")
print("Example classes:", sorted(os.listdir(train_dir))[:10])

Locating training directory...
✓ Training directory found: /content/simpsons_extracted/archive/characters_train
Example classes: ['.DS_Store', 'abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson', 'carl_carlson', 'charles_montgomery_burns', 'chief_wiggum', 'cletus_spuckler']


In [ ]:
# ============================================================================
# STEP 5: Define Transforms
# ============================================================================

IMG_SIZE = 64

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

print("✓ Transforms created")
print(f"  Normalization mean: {NORM_MEAN}")
print(f"  Normalization std:  {NORM_STD}")
print("  ⚠ Make sure inference.ipynb uses these exact same values!")

✓ Transforms created
  Normalization mean: [0.485, 0.456, 0.406]
  Normalization std:  [0.229, 0.224, 0.225]
  ⚠ Make sure inference.ipynb uses these exact same values!


In [ ]:
# ============================================================================
# STEP 6: Load Dataset
# ============================================================================
base_dataset = datasets.ImageFolder(train_dir)
classes = base_dataset.classes
num_classes = len(classes)

print("Total images:", len(base_dataset))
print("Number of classes:", num_classes)
print("First 10 classes:", classes[:10])

with open("classes.json", "w") as f:
    json.dump(classes, f, indent=2)

print("✓ classes.json saved")

Total images: 16764
Number of classes: 42
First 10 classes: ['abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson', 'carl_carlson', 'charles_montgomery_burns', 'chief_wiggum', 'cletus_spuckler', 'comic_book_guy']
✓ classes.json saved


In [ ]:
# ============================================================================
# STEP 7: Train / Validation Split
# ============================================================================
train_size = int(0.8 * len(base_dataset))
val_size = len(base_dataset) - train_size

generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(base_dataset, [train_size, val_size], generator=generator)

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
        self.classes = subset.dataset.classes

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        return image, label

train_dataset = TransformSubset(train_subset, train_transform)
val_dataset = TransformSubset(val_subset, val_transform)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Train images: 13411
Validation images: 3353


In [ ]:
# ============================================================================
# STEP 8: Create DataLoaders
# ============================================================================
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✓ DataLoaders created with batch size {BATCH_SIZE}")

✓ DataLoaders created with batch size 32


In [ ]:
# ============================================================================
# STEP 9: Define CNN Model From Scratch  (SimpsonsCNN)
# ============================================================================

class SimpsonsCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 32 -> 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 16 -> 8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 8 -> 4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpsonsCNN(num_classes).to(device)

print("Using device:", device)
print("Number of classes:", num_classes)
print("✓ Model created from scratch")

Using device: cuda
Number of classes: 42
✓ Model created from scratch


In [ ]:
# ============================================================================
# STEP 10: Loss, Optimizer, and Evaluation Function
# ============================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("✓ Training setup ready")

✓ Training setup ready


In [ ]:
# ============================================================================
# STEP 11: Train Model
# ============================================================================
EPOCHS = 25
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    scheduler.step()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "model.pth")
        print("  ✓ Best model saved as model.pth")

print("Training finished")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

Epoch [1/25], Batch [100/420], Loss: 2.9181
Epoch [1/25], Batch [200/420], Loss: 2.4466
Epoch [1/25], Batch [300/420], Loss: 2.2513
Epoch [1/25], Batch [400/420], Loss: 2.2711
Epoch [1/25]
  Train Loss: 2.4757 | Train Acc: 32.82%
  Val Loss:   1.8030 | Val Acc:   49.69%
  ✓ Best model saved as model.pth
Epoch [2/25], Batch [100/420], Loss: 1.7757
Epoch [2/25], Batch [200/420], Loss: 2.1478
Epoch [2/25], Batch [300/420], Loss: 1.1167
Epoch [2/25], Batch [400/420], Loss: 1.4229
Epoch [2/25]
  Train Loss: 1.7041 | Train Acc: 52.84%
  Val Loss:   1.2584 | Val Acc:   65.40%
  ✓ Best model saved as model.pth
Epoch [3/25], Batch [100/420], Loss: 1.5591
Epoch [3/25], Batch [200/420], Loss: 1.3396
Epoch [3/25], Batch [300/420], Loss: 1.4438
Epoch [3/25], Batch [400/420], Loss: 1.2238
Epoch [3/25]
  Train Loss: 1.3789 | Train Acc: 61.93%
  Val Loss:   1.2223 | Val Acc:   66.21%
  ✓ Best model saved as model.pth
Epoch [4/25], Batch [100/420], Loss: 0.8767
Epoch [4/25], Batch [200/420], Loss: 1.42

In [ ]:
# ============================================================================
# STEP 12: Save Final Model Artifacts
# ============================================================================
if not os.path.exists("model.pth"):
    torch.save(model.state_dict(), "model.pth")

shutil.copy("model.pth", "simpson_model.pth")

with open("classes.json", "w") as f:
    json.dump(classes, f, indent=2)

print("✓ Saved model.pth")
print("✓ Saved simpson_model.pth")
print("✓ Saved classes.json")
print("Files in current directory:")
print([f for f in os.listdir('.') if f.endswith(('.pth', '.json', '.ipynb'))])

✓ Saved model.pth
✓ Saved simpson_model.pth
✓ Saved classes.json
Files in current directory:
['simpson_model.pth', 'model.pth', 'classes.json']
